In [1]:
import json, re, time
from collections import Counter
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import sparse

# scikit-learn: só divisão dos dados, métricas, lista de stopwords e conferências
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# semente fixa: os resultados se repetem a cada execução
SEED = 42

DADOS = Path("articles.csv")

if not DADOS.exists():                      # kernel aberto em notebooks/ em vez de notebooks/classification/
    DADOS = Path("classification") / DADOS

FIG = Path("figuras"); 
FIG.mkdir(exist_ok=True)
RESULTADOS = {}                             # números-chave, salvos em resultados.json no final

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
CORES = [AZUL, LARANJA, VERDE, "#eda100", "#e87ba4"]
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False,
})


def salvar(fig, nome):
    # salva a figura em figuras/ (usada no relatório e nos slides)
    fig.tight_layout()
    fig.savefig(FIG / nome, bbox_inches="tight", facecolor="white")


def metricas(y, p):
    # métricas pedidas no enunciado (Aula 07; Géron cap. 3)
    return {"Precision (macro)": precision_score(y, p, average="macro", zero_division=0),
            "Recall (macro)": recall_score(y, p, average="macro", zero_division=0),
            "F1-micro": f1_score(y, p, average="micro"),
            "F1-macro": f1_score(y, p, average="macro"),
            "Acurácia": accuracy_score(y, p)}

print("configuração pronta")
print("Pasta do Python:", Path.cwd())
print(DADOS)

configuração pronta
Pasta do Python: /home/jovyan/work/ classification/notebooks/classification
articles.csv


In [2]:
dados = pd.read_csv(DADOS)
print(f"formato: {dados.shape}, | colunas: {list(dados.columns)}")
print(dados.category.value_counts().to_string())
print("\nexemplo de notícia:\n", dados.text.iloc[0][:300], "...")

formato: (167053, 6), | colunas: ['title', 'text', 'date', 'category', 'subcategory', 'link']
category
poder                           22022
colunas                         21622
mercado                         20970
esporte                         19730
mundo                           17130
cotidiano                       16967
ilustrada                       16345
opiniao                          4525
paineldoleitor                   4011
saopaulo                         3955
tec                              2260
tv                               2142
educacao                         2118
turismo                          1903
ilustrissima                     1411
ciencia                          1335
equilibrioesaude                 1312
sobretudo                        1057
bbc                               980
folhinha                          876
empreendedorsocial                841
comida                            828
asmais                            548
ambiente               

In [47]:
# Limpeza: duplicas e data leakage
"""
Data leakage acontece quando o modelo recebe, no treino, informação que não terá em produção. O resultado é um desempenho no 
teste artificialmente alto. Um caso clássico é a contaminação por amostragem: se a mesma notícia aparece duas vezes na base, 
uma cópia pode cair no treino e a outra no teste, e o modelo é avaliado em algo que já viu.
"""

resultados = dict()

# duplicated() devolve True para cada texto que já apareceu antes (a primeira ocorrência fica False). .sum() conta os True, 
# já que True vale 1. Resultado: quantas notícias são cópias repetidas.
n_dup = dados.text.duplicated().sum()

# Simular uma divisão nós dados de treinos sujos
_tv, _te = train_test_split(dados, test_size=0.20, stratify=dados.category, random_state=SEED)
leaker = _te.text.isin(set(_tv.text)).sum()
print(f"notícias duplicadas: {n_dup}")
print(f"sem a limpeza, {leaker} notícias do teste teriam uma cópia idêntica no treino")

print(f"Antes de remover duplicatas: {len(dados.text)}")

df = dados.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Após remover duplicatas: {len(df)} notícias")
resultados["dados"] = {"bruto": len(dados), "duplicatas": int(n_dup), "vazamento_evitado": int(leaker), "limpo": len(df)}
resultados


notícias duplicadas: 99
sem a limpeza, 32 notícias do teste teriam uma cópia idêntica no treino
Antes de remover duplicatas: 2225
Após remover duplicatas: 2126 notícias


{'dados': {'bruto': 2225,
  'duplicatas': 99,
  'vazamento_evitado': 32,
  'limpo': 2126}}

In [74]:
# Divisão hold-out 60/20/20 estratíficada.
# Treino    (60%): ajusta os parâmetros (pesos da regressão, cortes da árvore) e o vocabulário  do TF - IDF.
# Validação (20%): escolhe hiperparâmetros (alpha, profundidade) e desenha as curvas de aprendizado.
# Teste     (20%): Avaliação final, usada uma única vez por modelo
# Teste (20%): avaliação final, usada uma única vez por modelo.

# amostragem estratificada mantém a proporção das classes igual nos três conjuntos. Primeiro separâmos 20% para teste; do restante (80%), 25% vão para validaçãom
# ou seja, 20% do total;

treino_val, teste = train_test_split(df, test_size=0.20, stratify=df.category, random_state=SEED)
treino, val = train_test_split(treino_val, test_size=0.25, stratify=treino_val.category, random_state=SEED)
y_treino, y_val, y_te = treino.category.values,  val.category.values, teste.category.values


CLASSES = sorted(df.category.unique())
tabela = pd.DataFrame({"treino": treino.category.value_counts(), "validação": val.category.value_counts(),
                       "teste": teste.category.value_counts()}).reindex(CLASSES)

tabela.loc["total"] = tabela.sum()
print(tabela.to_string())
print("\nproporção de cada classe (%):")
print((tabela.drop("total") / tabela.drop("total").sum()  * 100).round(1).to_string())

treino_val.head()


               treino  validação  teste
category                               
business          301        101    101
entertainment     221         74     74
politics          242         80     81
sport             302        101    101
tech              209         69     69
total            1275        425    426

proporção de cada classe (%):
               treino  validação  teste
category                               
business         23.6       23.8   23.7
entertainment    17.3       17.4   17.4
politics         19.0       18.8   19.0
sport            23.7       23.8   23.7
tech             16.4       16.2   16.2


,category,text
987,politics,child access law shake-up planned parents who ...
1649,politics,assembly ballot papers missing hundreds of b...
597,business,pension hitch for long-living men male life ex...
1556,politics,final hunts held as ban looms hunts in england...
1202,sport,arsenal through on penalties arsenal win 4-2 o...


Os algoritmos trabalham com números, mas uma notícia é uma sequência de palavras de tamanho variável. O modelo *bag-of-words* transforma cada documento num vetor com um componente por termo do vocabulário.

A frequência pura do termo (TF) tem dois problemas apontados na Aula 06: um termo frequente em todos os documentos (como *"said"*) recebe peso alto sem discriminar nada, e um termo raro no corpus deveria valer mais. O IDF corrige isso:

$$
\mathrm{idf}(t) = 1 + \ln\frac{n}{n_t}
$$

$$
\mathrm{tfidf}(d,t) = \frac{\mathrm{tf}(d,t)\cdot\mathrm{idf}(t)}{\sqrt{\sum_j \big(\mathrm{tf}(d,t_j)\cdot\mathrm{idf}(t_j)\big)^2}}
$$

onde $n$ é o número de documentos e $n_t$ o número de documentos que contêm o termo $t$. A divisão final (norma L2) compensa o tamanho diferente das notícias.

In [5]:
corpus = ["Ambiente agradável e tranquilo. Comida e música com qualidade. Adoramos o filé à Parmegiana.",
          "O Filé à Parmegiana da cidade. Ambiente agradável e qualidade no atendimento",
          "O Filé à Parmegiana com fritas é uma delícia."]

stop_pt = {"e", "o", "a", "à", "com", "da", "no", "é", "uma"}

def stop_words(corpus: list[str], stop_pt: set[str]):
    docs = []
    
    for doc in corpus:
        text = doc.lower()
        words = re.findall("\w+", text)
        tokens: list[str] = []
        
        for word in words:
            if word not in stop_pt:
                tokens.append(word)
        docs.append(tokens)

    return docs

def pick_up_vocabulary(): 
    pass 
    
def word_count_in_document():
    pass

def calculate_idf():
    pass


# \w corresponnde a uníco caractere que pode fazer parte de uma palavra:
# letras: a-z, A-Z, e também acentuadas como é, ã
# digitos 0-0
# O sublinhado _

# + diz "repita o que vem antes uma ou mais vezes, pegando o máximo possivel"

docs = stop_words(corpus, stop_pt)
print(docs)

vocab_ex = set()

for doc in docs:
    for word in doc:
        vocab_ex.add(word)

vocab_ex = sorted(vocab_ex)

n_t = {}

for w in vocab_ex:
    cont: int = 0
    for doc in docs:
        if w in doc:
            n_t[w] = n_t.get(w, 0) + 1
      
n_t_array = np.array(list(n_t.values()))

idf_list = []

n = len(docs)

for cont in n_t_array:
    value = 1 + math.log(n / cont)
    idf_list.append(value)

    

[['ambiente', 'agradável', 'tranquilo', 'comida', 'música', 'qualidade', 'adoramos', 'filé', 'parmegiana'], ['filé', 'parmegiana', 'cidade', 'ambiente', 'agradável', 'qualidade', 'atendimento'], ['filé', 'parmegiana', 'fritas', 'delícia']]
